# Cleaning the Data

## Init

In [0]:
from pyspark.sql.functions import col

## Reading from Bronze Schema

In [0]:
pitching_df = spark.table('pitch_data_2026.bronze.pitch_data')
# Temp view for SQL
pitching_df.createOrReplaceTempView("pitching_df_sql")

## Filter Columns

In [0]:
pitching_select = spark.sql(
    """
    SELECT game_date, pitcher, player_name, pitch_name, release_speed, release_spin_rate, zone, balls, strikes,
    inning, inning_topbot, at_bat_number, pitch_number, attack_zone, events, description, age_pit_legacy, p_throws, pitch_type, pfx_x, pfx_z, plate_x, plate_z, game_pk, spin_axis, api_break_z_with_gravity, api_break_x_arm
    FROM pitching_df_sql
    """
    )

## Null Values

In [0]:
# Get Nulls function
def get_nulls(df):
    nulls = {}
    for c in df.columns:
        null_count = df.filter(col(c).isNull()).count()
        if null_count > 0:
            nulls[c] = null_count
    return nulls

# Checking which columns have null values
print('Null values:', get_nulls(pitching_select))

# The Null values in every column, besides events, is negligable. 
# Those results are due to collection errors on MLB's behalf and will be kept as nulls

# For the events column, Null values represent if there was no field result after the pitch (excluding stikeouts and walks)
# Every null values for events will be replaced with 'no_field_event'

pitching_select = pitching_select.na.fill({'events': 'no_field_event'})

## Total Pitches

In [0]:
pitching_select.createOrReplaceTempView("pitching_select")

# Creating total_pitch_count by sorting accordingly and partitioning by the game id and name of the pitcher

pitching_select = spark.sql(
    """
    SELECT * ,
    ROW_NUMBER() OVER (
        PARTITION BY game_pk, pitcher
        ORDER BY game_pk, pitcher, inning, at_bat_number, pitch_number
        ) AS total_pitch_count
        FROM pitching_select
    """
    )

# Dropping at_bat_number and pitch_number (no longer necessary)
pitching_select = pitching_select.drop("at_bat_number", "pitch_number")

## Writing into Silver Schema

In [0]:
pitching_select.write.mode("overwrite").saveAsTable("pitch_data_2026.silver.pitch_data")